# Shared LSC Context Audit

This notebook audits the shared mention-level context table created by `01_build_lsc_contexts.ipynb`. It does not rebuild contexts. Its role is to verify that the handoff table is structurally sound, surface coverage and domain-concentration risks, and prepare a compact manual-inspection handoff.

## Setup

The hard checks below catch problems that would make downstream LSC notebooks unsafe to run. Warning flags identify issues that need interpretation rather than immediate failure.

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "configs/commoncrawl_collection.yaml"
CONTEXT_DIR = PROJECT_ROOT / "data/interim/lsc/contexts"

CONTEXT_PATH = CONTEXT_DIR / "lsc_mention_contexts.parquet"
COUNTS_BY_UNIT_PATH = CONTEXT_DIR / "lsc_context_counts_by_year_unit.csv"
COUNTS_BY_RAW_PATH = CONTEXT_DIR / "lsc_context_counts_by_year_unit_raw_form.csv"
TOP_DOMAINS_PATH = CONTEXT_DIR / "lsc_context_top_domains_by_year_unit.csv"
MANUAL_SAMPLE_PATH = CONTEXT_DIR / "lsc_context_manual_samples.csv"
EXTRACTION_SUMMARY_PATH = CONTEXT_DIR / "lsc_context_extraction_summary.csv"
PUBLICATION_EXCLUSIONS_PATH = CONTEXT_DIR / "lsc_context_publication_date_exclusions.csv"
AUDIT_CHECKS_PATH = CONTEXT_DIR / "lsc_context_audit_checks.csv"
AUDIT_FLAGS_PATH = CONTEXT_DIR / "lsc_context_audit_flags.csv"

EXPECTED_UNITS = ["ADHD", "Autism", "frustration", "loneliness", "sadness"]
MAX_MENTIONS_PER_DOC_UNIT = 3
LOW_DOCUMENT_WARN_THRESHOLD = 50
TOP_DOMAIN_SHARE_WARN_THRESHOLD = 0.20
MIN_MANUAL_SAMPLES_PER_UNIT_YEAR = 1

with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

collection_window = config["collection"]["window"]
EXPECTED_YEARS = list(range(collection_window["primary_start_year"], collection_window["end_year"] + 1))

PROJECT_ROOT

## Load Builder Outputs

Most diagnostics are read from the builder outputs. The parquet table is loaded with only the columns needed for structural checks, so the audit stays lighter than the context-building notebook.

In [ ]:
required_paths = {
    "context_table": CONTEXT_PATH,
    "counts_by_unit": COUNTS_BY_UNIT_PATH,
    "counts_by_raw": COUNTS_BY_RAW_PATH,
    "top_domains": TOP_DOMAINS_PATH,
    "manual_samples": MANUAL_SAMPLE_PATH,
    "extraction_summary": EXTRACTION_SUMMARY_PATH,
    "publication_exclusions": PUBLICATION_EXCLUSIONS_PATH,
}
missing_paths = [name for name, path in required_paths.items() if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing LSC context output(s): {missing_paths}")

context_columns = [
    "doc_id",
    "lsc_year",
    "published_year",
    "source_year",
    "analysis_unit",
    "raw_form",
    "collapsed_raw_forms",
    "collapsed_match_count",
    "acronym_expansion_collapsed",
    "registered_domain",
    "mention_start_char",
    "mention_end_char",
    "target_sentence",
    "token_window_5",
    "cap_applied",
]
contexts = pd.read_parquet(CONTEXT_PATH, columns=context_columns)
counts_by_unit = pd.read_csv(COUNTS_BY_UNIT_PATH)
counts_by_raw = pd.read_csv(COUNTS_BY_RAW_PATH)
top_domains = pd.read_csv(TOP_DOMAINS_PATH)
manual_samples = pd.read_csv(MANUAL_SAMPLE_PATH)
extraction_summary = pd.read_csv(EXTRACTION_SUMMARY_PATH)
publication_exclusions = pd.read_csv(PUBLICATION_EXCLUSIONS_PATH)

print(f"Context rows: {len(contexts):,}")
print(f"Documents: {contexts['doc_id'].nunique():,}")
print(f"Analysis units: {', '.join(sorted(contexts['analysis_unit'].dropna().unique()))}")

## Normalise Diagnostic Tables

The builder now writes top-domain ranks and shares, but this cell also supports earlier top-domain files that contain only raw counts. That makes the audit robust when rerun after partial or interrupted preprocessing.

In [ ]:
top_domains = top_domains.copy()
sort_columns = ["lsc_year", "analysis_unit", "mentions", "registered_domain"]
top_domains = top_domains.sort_values(sort_columns, ascending=[True, True, False, True]).reset_index(drop=True)

if "domain_rank" not in top_domains.columns:
    top_domains["domain_rank"] = top_domains.groupby(["lsc_year", "analysis_unit"]).cumcount() + 1

if "total_mentions_for_year_unit" not in top_domains.columns:
    denominators = counts_by_unit[["lsc_year", "analysis_unit", "mentions"]].rename(columns={"mentions": "total_mentions_for_year_unit"})
    top_domains = top_domains.merge(denominators, on=["lsc_year", "analysis_unit"], how="left")

if "mention_share" not in top_domains.columns:
    top_domains["mention_share"] = top_domains["mentions"] / top_domains["total_mentions_for_year_unit"]

top_domains.head()

## Hard Structural Checks

In [ ]:
checks: list[dict[str, object]] = []

def add_check(name: str, value: object, passes: bool, severity: str = "fail") -> None:
    checks.append({"check": name, "value": value, "passes": bool(passes), "severity": severity})

observed_units = sorted(contexts["analysis_unit"].dropna().unique())
observed_years = sorted(contexts["lsc_year"].dropna().astype(int).unique())
missing_units = sorted(set(EXPECTED_UNITS) - set(observed_units))
unexpected_units = sorted(set(observed_units) - set(EXPECTED_UNITS))
missing_years = sorted(set(EXPECTED_YEARS) - set(observed_years))
unexpected_years = sorted(set(observed_years) - set(EXPECTED_YEARS))

max_mentions_per_doc_unit = int(contexts.groupby(["doc_id", "analysis_unit"]).size().max())
duplicate_mentions = int(contexts.duplicated(["doc_id", "analysis_unit", "raw_form", "mention_start_char", "mention_end_char"]).sum())
missing_target_sentence = int(contexts["target_sentence"].fillna("").str.strip().eq("").sum())
missing_token_window = int(contexts["token_window_5"].fillna("").str.strip().eq("").sum())
invalid_offsets = int((contexts["mention_end_char"] <= contexts["mention_start_char"]).sum())
missing_lsc_year = int(contexts["lsc_year"].isna().sum())
missing_published_year = int(contexts["published_year"].isna().sum())
lsc_year_mismatch = int((contexts["lsc_year"] != contexts["published_year"]).sum())
published_year_outside_window = int((~contexts["published_year"].between(min(EXPECTED_YEARS), max(EXPECTED_YEARS))).sum())
missing_collapsed_raw_forms = int(contexts["collapsed_raw_forms"].fillna("").str.strip().eq("").sum())
missing_collapsed_match_count = int(contexts["collapsed_match_count"].isna().sum())
summary_metrics = dict(zip(extraction_summary["metric"], extraction_summary["value"]))
collapse_arithmetic_expected = int(summary_metrics.get("matches_after_overlap_resolution", 0)) - int(summary_metrics.get("acronym_expansion_mentions_removed", 0))
collapse_arithmetic_observed = int(summary_metrics.get("matches_after_acronym_expansion_collapse", -1))
unit_count_total = int(counts_by_unit["mentions"].sum())
raw_count_total = int(counts_by_raw["mentions"].sum())
publication_statuses = set(publication_exclusions["publication_date_status"].dropna())

add_check("context_rows_positive", len(contexts), len(contexts) > 0)
add_check("expected_analysis_units_present", ", ".join(observed_units), not missing_units)
add_check("no_unexpected_analysis_units", ", ".join(unexpected_units), not unexpected_units, severity="warn")
add_check("expected_years_present", ", ".join(map(str, observed_years)), not missing_years)
add_check("no_unexpected_years", ", ".join(map(str, unexpected_years)), not unexpected_years, severity="warn")
add_check("mention_cap_respected", max_mentions_per_doc_unit, max_mentions_per_doc_unit <= MAX_MENTIONS_PER_DOC_UNIT)
add_check("duplicate_mention_rows", duplicate_mentions, duplicate_mentions == 0)
add_check("target_sentence_non_empty", missing_target_sentence, missing_target_sentence == 0)
add_check("token_window_non_empty", missing_token_window, missing_token_window == 0)
add_check("valid_mention_offsets", invalid_offsets, invalid_offsets == 0)
add_check("lsc_year_non_empty", missing_lsc_year, missing_lsc_year == 0)
add_check("published_year_non_empty", missing_published_year, missing_published_year == 0)
add_check("lsc_year_equals_published_year", lsc_year_mismatch, lsc_year_mismatch == 0)
add_check("published_year_in_lsc_window", published_year_outside_window, published_year_outside_window == 0)
add_check("publication_exclusions_written", ", ".join(sorted(publication_statuses)), "kept_for_lsc_year" in publication_statuses)
add_check("collapsed_raw_forms_recorded", missing_collapsed_raw_forms, missing_collapsed_raw_forms == 0)
add_check("collapsed_match_count_recorded", missing_collapsed_match_count, missing_collapsed_match_count == 0)
add_check("collapse_summary_arithmetic", collapse_arithmetic_observed, collapse_arithmetic_observed == collapse_arithmetic_expected)
add_check("unit_counts_match_context_rows", unit_count_total, unit_count_total == len(contexts))
add_check("raw_counts_match_context_rows", raw_count_total, raw_count_total == len(contexts))

audit_checks = pd.DataFrame(checks)
audit_checks.to_csv(AUDIT_CHECKS_PATH, index=False)
audit_checks

## Coverage Diagnostics

Low sample sizes are warning flags, not automatic exclusions. Downstream notebooks should decide whether to pool, smooth, or report uncertainty for low-volume unit-years.

In [ ]:
coverage_mentions = counts_by_unit.pivot(index="lsc_year", columns="analysis_unit", values="mentions").reindex(columns=EXPECTED_UNITS).fillna(0).astype(int)
coverage_documents = counts_by_unit.pivot(index="lsc_year", columns="analysis_unit", values="documents").reindex(columns=EXPECTED_UNITS).fillna(0).astype(int)

low_coverage = counts_by_unit.loc[counts_by_unit["documents"] < LOW_DOCUMENT_WARN_THRESHOLD, ["lsc_year", "analysis_unit", "documents", "mentions"]].copy()
coverage_mentions

In [ ]:
coverage_documents

## Domain-Concentration Diagnostics

A high top-domain share can indicate genuine discourse concentration or residual crawler/template artefacts. These cases should be inspected before interpreting a year-specific spike.

In [ ]:
top_domain_rows = top_domains.loc[top_domains["domain_rank"] == 1, ["lsc_year", "analysis_unit", "registered_domain", "mentions", "documents", "mention_share"]].copy()
domain_concentration_flags = top_domain_rows.loc[top_domain_rows["mention_share"] >= TOP_DOMAIN_SHARE_WARN_THRESHOLD].sort_values("mention_share", ascending=False)
top_domain_rows.sort_values("mention_share", ascending=False).head(20)

## Raw-Form Balance

Raw-form totals are diagnostic only. The main target analysis remains at conceptual group level for ADHD and Autism.

In [ ]:
raw_form_totals = (
    counts_by_raw.groupby(["analysis_unit", "raw_form"], as_index=False)
    .agg(documents=("documents", "sum"), mentions=("mentions", "sum"))
    .sort_values(["analysis_unit", "mentions"], ascending=[True, False])
)
raw_form_totals

## Manual Sample Handoff

These rows are for qualitative inspection. They are not used to produce automatic exclusions in this notebook.

In [ ]:
sample_counts = manual_samples.groupby(["lsc_year", "analysis_unit"]).size().reset_index(name="sample_rows")
expected_sample_keys = counts_by_unit[["lsc_year", "analysis_unit"]].drop_duplicates()
sample_coverage = expected_sample_keys.merge(sample_counts, on=["lsc_year", "analysis_unit"], how="left").fillna({"sample_rows": 0})
missing_samples = sample_coverage.loc[sample_coverage["sample_rows"] < MIN_MANUAL_SAMPLES_PER_UNIT_YEAR].copy()

sample_columns = [
    "lsc_year",
    "published_year",
    "source_year",
    "analysis_unit",
    "raw_form",
    "collapsed_raw_forms",
    "collapsed_match_count",
    "acronym_expansion_collapsed",
    "registered_domain",
    "target_sentence",
    "url",
]
manual_samples[sample_columns].head(30)

## Audit Flags and Verdict

In [ ]:
flags: list[dict[str, object]] = []

for _, row in audit_checks.loc[~audit_checks["passes"]].iterrows():
    flags.append({"severity": row["severity"], "category": "structural_check", "lsc_year": None, "analysis_unit": None, "detail": row["check"], "value": row["value"]})

for _, row in low_coverage.iterrows():
    flags.append({"severity": "warn", "category": "low_document_count", "lsc_year": int(row["lsc_year"]), "analysis_unit": row["analysis_unit"], "detail": "documents below warning threshold", "value": int(row["documents"])})

for _, row in domain_concentration_flags.iterrows():
    flags.append({"severity": "warn", "category": "top_domain_concentration", "lsc_year": int(row["lsc_year"]), "analysis_unit": row["analysis_unit"], "detail": row["registered_domain"], "value": float(row["mention_share"])})

for _, row in missing_samples.iterrows():
    flags.append({"severity": "warn", "category": "manual_sample_missing", "lsc_year": int(row["lsc_year"]), "analysis_unit": row["analysis_unit"], "detail": "manual sample rows below minimum", "value": int(row["sample_rows"])})

audit_flags = pd.DataFrame(flags, columns=["severity", "category", "lsc_year", "analysis_unit", "detail", "value"])
audit_flags.to_csv(AUDIT_FLAGS_PATH, index=False)

hard_failures = audit_flags.loc[audit_flags["severity"] == "fail"]
if not hard_failures.empty:
    raise AssertionError(f"Shared LSC context audit failed hard checks: {hard_failures['detail'].tolist()}")

print(f"Wrote {AUDIT_CHECKS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {AUDIT_FLAGS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Warnings: {(audit_flags['severity'] == 'warn').sum() if not audit_flags.empty else 0}")
print("Shared LSC context audit passed hard checks.")
audit_flags.head(30)